In [1]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI
import gradio as gr # oh yeah!

# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [2]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [3]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [4]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [5]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [6]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [7]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in without markdown.
Include details of company culture, customers and careers/jobs if you have the information.
"""

def stream_brochure(company_name, url):
    print(company_name, url)
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    ) 
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        print(result)
        yield result
   

In [ ]:

name_input = gr.Textbox(label="Company name:")
url_input = gr.Textbox(label="Landing page URL including http:// or https://")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_brochure,
    title="AI Brochure Generator", 
    inputs=[name_input, url_input], 
    outputs=[message_output], 
    examples=[
            ["OpenAi", "https://openai.com/"],
        ], 
    flagging_mode="never"
    )
view.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://22f651e257914b986d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


OpenAi https://openai.com/

Open
OpenAI
OpenAI Bro
OpenAI Brochure
OpenAI Brochure


OpenAI Brochure

About
OpenAI Brochure

About Open
OpenAI Brochure

About OpenAI
OpenAI Brochure

About OpenAI  

OpenAI Brochure

About OpenAI  
Open
OpenAI Brochure

About OpenAI  
OpenAI
OpenAI Brochure

About OpenAI  
OpenAI is
OpenAI Brochure

About OpenAI  
OpenAI is a
OpenAI Brochure

About OpenAI  
OpenAI is a leading
OpenAI Brochure

About OpenAI  
OpenAI is a leading AI
OpenAI Brochure

About OpenAI  
OpenAI is a leading AI research
OpenAI Brochure

About OpenAI  
OpenAI is a leading AI research and
OpenAI Brochure

About OpenAI  
OpenAI is a leading AI research and deployment
OpenAI Brochure

About OpenAI  
OpenAI is a leading AI research and deployment company
OpenAI Brochure

About OpenAI  
OpenAI is a leading AI research and deployment company dedicated
OpenAI Brochure

About OpenAI  
OpenAI is a leading AI research and deployment company dedicated to
OpenAI Brochure

About OpenAI  
OpenA